In [ ]:
# imports
import rasterio, re, pandas as pd, geopandas as gpd, numpy as np
from pathlib import Path
from rasterio.mask import mask
from scipy.spatial import distance_matrix

In [ ]:
class Banana_Census:
    
    def __init__(self, in_tiff_path, in_dsm_path, in_csv_path, in_boundary_path):

        paths = ["image_path", "dsm_path", "csv_path", "boundary_path"]
        inputs = [in_tiff_path, in_dsm_path, in_csv_path, in_boundary_path]

        for attr, input_path in zip(paths, inputs):
            setattr(self, attr, Path(input_path))

        # input directory validation
        if not self.csv_path.exists() and self.image_path.exists():
            raise FileNotFoundError(f"one or both directories '{self.csv_path}', and '{self.image_path}' does not exist.")
            return
            
    # iterate and pair image - csv
    def image_csv_pair (self, img_cvs_dir):

        # get files from both image and csv parent directories
        files_by_subdir = {
            sub_dir.name: [list(pair) for pair in zip(*[
                sorted([file.name for file in parent_dir.glob(f"{sub_dir.name}/*") if file.is_file()])
                for parent_dir in img_cvs_dir
            ])]
            for sub_dir in img_cvs_dir[0].glob("*") if sub_dir.is_dir()
        }
        # produce a dictionary of image - csv pair
        return files_by_subdir
    
    
    # gets the block id number
    def extract_number(self, word):
        # find the first occurrence of digits
        match = re.search(r'\d+', word)
        # Convert to integer if found
        return int(match.group()) if match else None


    # match number of image tiles to csv's
    def img_csv_match (self):        
        if len(list(self.csv_path.iterdir())) == len(list(self.image_path.iterdir())):
            
            # validate same items in both csv and image dir
            all_csvs = {csv.stem for csv in self.csv_path.iterdir()}
            all_imgs = {img.stem for img in self.image_path.iterdir()}

            # Assert that both sets contain the same items
            assert all_imgs == all_csvs, f"Items do not match. Missing image files: {all_csvs - all_imgs}, Missing CSV files: {all_imgs - all_csvs}"
       
        else: print('csv and image datasets do not have the same number of items.')


    # use boundary to clip points
    def aoi_gdf(self, in_gdf, in_bdry_path):

        #  initialize clipped_gdf
        clipped_gdf = None

        # iterate all block boundaries
        for block_boundary in in_bdry_path.iterdir():
            boundary = gpd.read_file(block_boundary)
        
            if in_gdf.crs != boundary.crs:
                boundary = boundary.to_crs(in_gdf.crs)

            # get which points are within the boundary
            within_mask = in_gdf.geometry.within(boundary.union_all())

            # cal the percentage of points within boundary
            within_ratio = within_mask.mean()

            # if 95% of points fall inside the boundary clip
            if within_ratio >= 0.95:
                clipped_gdf = in_gdf[within_mask].copy()

        # Handle case where no boundary meets 95% condition
        if clipped_gdf is None:
            raise ValueError("No boundary contains at least 95% of the points.")

        return clipped_gdf
    
    
    # extract image indices using mask
    def image_values(self, image, row_geometry):

        # mask the raster with the polygon
        out_image, out_transform = mask(image, [row_geometry], crop=True)

        # get the first band (if multi-band)
        out_image = out_image[0]

        # extract the DN values within the polygon
        dn_values = out_image.flatten()
        pos_dn_values = dn_values[dn_values > 0]

        # get the coordinates of the pixels
        rows, cols = np.indices(out_image.shape)
        xs, ys = rasterio.transform.xy(out_transform, rows, cols)

        # convert to arrays
        xs = np.array(xs)
        ys = np.array(ys)

        return dn_values, pos_dn_values, xs, ys
    
    
    # get pixel distance from centroid
    def dn_distance(self, row_geometry, all_x, all_y, dn_val):

        # centroid of the polygon
        centroid = row_geometry.centroid

        # compute distance from each pixel to the centroid
        distances = np.sqrt((all_x - centroid.x)**2 + (all_y - centroid.y)**2)

        # flatten arrays
        distances = distances.flatten()

        # use +dn_values to filter dn_distances
        pos_distances = distances[dn_val > 0]

        return pos_distances
    
    
    # locate central pixels (pixels within distance threshold)
    def percentile_filter(self, dn_distance, percentile_num, dn_vals):
        
        # set limit to a percentile of closest pixels
        threshold_distance = np.percentile(dn_distance, percentile_num) 
        
        # filter threshold pixels
        central_pixs = dn_vals[dn_distance <= threshold_distance]
        
        return central_pixs
    
    
    # compute weighted mean based on proximity to the centroid
    def mean_dn(self, central_pxls):

        # Adding a small value to avoid division by zero and normalize
        weights = 1 / (central_pxls + 1e-10)  
        normalized_weights = weights / np.sum(weights)

        # avearge dn
        weighted_mean_dn = np.average(central_pxls.flatten(), weights=normalized_weights.flatten())
        weighted_mean_dn = round(weighted_mean_dn,7)

        if weighted_mean_dn < 0:
            print(f"Weighted Mean DN value (central pixels): {weighted_mean_dn}")

        return weighted_mean_dn
    

    
    def pair_distance (self, pt_gdf):

        close_pairs = []

        # Convert geometries to coordinate arrays
        coords = np.array([[geom.x, geom.y] for geom in pt_gdf.geometry])

        # Compute pairwise distance matrix
        dist_matrix = distance_matrix(coords, coords)
        
        # Get upper triangular indices (to avoid duplicate comparisons)
        i_idx, j_idx = np.triu_indices(len(pt_gdf), k=1)

        # Print or save results
        for i, j in zip(i_idx, j_idx):
            if dist_matrix[i, j] < 1.5:
                close_pairs.append((i, j, pt_gdf.iloc[i]['Surface_value'], pt_gdf.iloc[j]['Surface_value']))

        return close_pairs


    def status (self, pt_gdf, plant_pairs):
        # Initialize all points as 'Parent' by default
        pt_gdf['status'] = 'Parent'

        # Assign 'Sucker' to the lower height in each pair
        for i, j, h1, h2 in plant_pairs:
            pt_gdf.at[j if h1 > h2 else i, 'status'] = 'Sucker'

        return pt_gdf
    

    # classify into parent / sucker
    def parent_sucker (self, point_gdf, dsm_file):

        # buffer each point to 0.5 meter radius
        point_gdf['geometry_buf'] = point_gdf.geometry.buffer(0.5)

        # read the dsm image 
        with rasterio.open(dsm_file) as dsm:

            # iterate buffered point_gdf
            for idx, row in point_gdf.iterrows():

                # extract dsm of 50th percentile around 0.5m
                dn_values, pos_dn, xs, ys = self.image_values(dsm, row['geometry_buf'])

                pos_distances = self.dn_distance(row['geometry_buf'], xs, ys, dn_values)
                
                central_pixels = self.percentile_filter(pos_distances, 50, pos_dn)
                
                weighted_mean_dn = self.mean_dn(central_pixels)

                point_gdf.at[idx, 'Surface_value'] = weighted_mean_dn
        
        point_gdf = point_gdf.drop(columns=['geometry_buf'])

        pairs = self.pair_distance(point_gdf)

        point_gdf = self.status(point_gdf, pairs)

        return point_gdf


    # read files create points
    def generate_geolocators (self, img_csv_pairs):

        # get all block points
        all_blocks_list = []

        for block_name, img_csv_pair in img_csv_pairs.items():

            block_list = []

            block_id = self.extract_number(block_name)

            for img_csv in img_csv_pair:
                
                csv_file = f'{self.csv_path}/{block_name}/{img_csv[1]}'

                df = pd.read_csv(csv_file)

                # get the label as numbers
                df['Plant_id'] = list(range(len(df)))

                # get the center points
                center_y = df['keypoint_x_center']
                center_x = df['keypoint_y_center']
                
                # get and image file
                image_file = f'{self.image_path}/{block_name}/{img_csv[0]}'

                # convert to geo-coordinates
                with rasterio.open(image_file) as image:
                    img_crs = image.crs
                    center = image.xy(center_x, center_y)

                # assign coordinates
                df['lat'], df['long'] = (center[1], center[0])

                # set block id
                df['block_id'] = block_id

                # keep necessary files
                df = df[['block_id','Plant_id','long','lat','elevation_status']]

                # Append gdf to block_list
                block_list.append(df)

            block_df = pd.concat(block_list, ignore_index=True)

            # create geodataframe
            gdf = gpd.GeoDataFrame(block_df, geometry=gpd.points_from_xy(block_df['long'], block_df['lat']), crs=img_crs)
            gdf = gdf.drop_duplicates(subset='geometry')

            clipped_gdf = self.aoi_gdf(gdf, self.boundary_path)

            # get the Parent / sucker
            dsm_file = f'{self.dsm_path}/{block_name}.tif'
            p_s_gdf = self.parent_sucker(clipped_gdf, dsm_file)

            # Append gdf to all_block_list
            all_blocks_list.append(p_s_gdf)

        return all_blocks_list  
       

    # Implementation
    def process(self):

        # check input csv and image file
        self.img_csv_match()

        # Define the parent directories
        img_cvs_dir = [self.image_path, self.csv_path]
        image_csv_pairs = self.image_csv_pair(img_cvs_dir)

        final_gdf = self.generate_geolocators(image_csv_pairs)

        # merge all gdf_list and remove duplicate
        merged_gdf = gpd.GeoDataFrame(pd.concat(final_gdf, ignore_index=True))
        merged_gdf = merged_gdf.drop_duplicates(subset='geometry')

        # assign new_id to the plants
        merged_gdf['Plant_id'] = (merged_gdf.index + 1).astype(str)

        # create a point_loc
        points_path = self.image_path.resolve().parent.joinpath('count_geo_output')   
        points_path.mkdir(parents=True, exist_ok=True)

        # save the merged_df as geojson file
        merged_gdf.to_file(f'{points_path}/model_points.geojson', driver='GeoJSON')

        # indicate points completion
        print(f'{points_path} successfully completed.')
            


# Data repository
input_image_dir= '.../Project_repo/count/count_image_tiles/' # imagery directory
input_dsm_dir='.../Project_repo/count/DSM_tiles/' # dsm blocks dir
input_csv_dir = '.../Project_repo/count/count_ML_output/' # model csv output directory
input_boundary_dir = '.../Project_repo/boundary_data/geojson_data/' # location of boundary data


# Implementation
post_processing = Banana_Census(input_image_dir, input_dsm_dir, input_csv_dir, input_boundary_dir)
post_processing.process()